# 116. 用户消费价值预测项目

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 31 / 34 步：把完整流程迁移到真实项目**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 模型保存与批量推理  →  **本章任务：** 用户消费价值预测项目  →  **下一步：** 物流延期风险预测项目
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

一个网络零售商每天都在产生海量订单——谁买了什么、买了多少、花了多少钱。



## 本章目标

学完本章，你将能够：

- **理解**：理解「用户消费价值预测项目」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「用户消费价值预测项目」的关键输出指标。
- **迁移**：能把「用户消费价值预测项目」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 116.1 数据字典

**背景引入**：一个网络零售商每天都在产生海量订单——谁买了什么、买了多少、花了多少钱。要判断一位客户接下来还会不会继续消费、大约会再花多少钱，第一步不是急着建模，而是先把这批原始交易记录翻查清楚：里面常常混着重复订单、取消单、缺失的客户编号和异常价格。这份课程所用的 UCI 线上零售数据正是这样的真实交易明细，先把它审计干净，后面的客户价值预测才有可靠的基础。


| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| InvoiceNo | 发票号 | C开头通常为取消单 |
| InvoiceDate | 交易时间 | 划分观察窗口与未来窗口 |
| CustomerID | 客户编号 | 聚合到一位客户一行 |
| Quantity/UnitPrice | 数量/单价 | 构造有效交易金额 |
| future_revenue | 未来消费金额 | 回归目标 |

## 116.2 数据质量检查清单

- 重复、取消、退货和非正价格
- CustomerID缺失和清洗保留率
- 观察窗口与目标窗口严格分离（打个比方：判断客户“以后还会不会再来”，只能用当下就看得到的信息，不能拿“以后才发生的消费”当线索——否则等于偷看答案。）
- 客户主键唯一
- 未来零消费比例和金额长尾
- 测试集不参与模型选择


## 116.3 项目任务

1. 明确客户粒度、预测时点和未来窗口
2. 审计原始交易数据
3. 清洗交易并记录样本变化
4. 构造历史特征和未来目标
5. 探索目标分布并建立Dummy基线
6. 划分客户训练集与测试集
7. 比较Ridge与梯度提升模型
8. 在原始金额尺度和Top-K上评价
9. 分析错误样本与客户分组
10. 解释特征重要性并总结模型局限


## 116.4 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 原始数据质量审计 | `pd.read_csv()`、`pd.Series()`、`raw.duplicated()`、`raw.CustomerID.isna()` | 先量化重复、取消、缺失和异常数值，建立可追溯的数据起点。 | 重复、取消、退货和非正价格 |
| 2. 清洗交易并记录样本变化 | `raw.drop_duplicates()`、`dedup.InvoiceNo.astype()`、`str.startswith()`、`dedup.CustomerID.notna()` | 删除重复后，仅保留有客户编号的有效正向销售，并计算清洗保留率。 | CustomerID缺失和清洗保留率 |
| 3. 定义时间窗口并构造客户样本 | `sales.InvoiceDate.quantile()`、`hist.groupby()`、`x.max()`、`x.min()` | 截止日前的数据只生成特征，截止日后的数据只生成目标，从源头防止时间穿越。 | 观察窗口与目标窗口严格分离 |
| 4. 探索目标分布与客户差异 | `customer.future_revenue.describe()`、`pd.qcut()`、`customer.groupby()`、`group_summary.round()` | 金额目标同时具有大量零值和明显长尾，后续需要在对数尺度训练、原始金额尺度评价。 | 客户主键唯一 |
| 5. 客户级划分与Dummy基线 | `np.log1p()`、`.clip()`、`.fit()`、`.mean()` | 按客户划分训练集和测试集，并用中位数回归器建立最低比较基线。 | 未来零消费比例和金额长尾 |
| 6. 比较线性模型与树模型 | `models.items()`、`rows.append()`、`scores.mean()`、`scores.std()` | 使用五折交叉验证比较Ridge与梯度提升，以对数目标MAE选择模型。 | 测试集不参与模型选择 |
| 7. 在原始金额尺度评价模型 | `np.maximum()`、`np.expm1()`、`best_model.predict()`、`dummy.predict()` | 把预测还原为金额，联合报告MAE、RMSE、R²和Dummy基线误差。 | 重复、取消、退货和非正价格 |
| 8. 使用Top-K检查排序能力 | `result.sort_values()`、`ranked.actual.sum()`、`topk_rows.append()`、`ranked.head()` | Top-K只作为模型排序评价，比较不同观察比例覆盖了多少真实未来消费金额。 | CustomerID缺失和清洗保留率 |
| 9. 错误切片与高误差案例 | `result.copy()`、`np.where()`、`pd.qcut()`、`error_table.groupby()` | 分别检查未来零消费/有消费客户，以及不同历史价值组的误差。 | 观察窗口与目标窗口严格分离 |
| 10. 特征解释与模型局限 | `pd.Series()`、`importance.round()`、`.sort_values()` | 置换重要性说明模型依赖哪些历史信号，不能据此断言这些变量会导致未来消费。 | 客户主键唯一 |


## 116.5 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 116.6 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 116.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 116.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 116.9 原始数据质量审计

先量化重复、取消、缺失和异常数值，建立可追溯的数据起点。


<!-- math-foundation:chapter-116 -->
### 数学推导｜客户价值预测的误差口径

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜为每位客户计算绝对误差。** $e_i=|y_i-\hat y_i|$。

**第 2 步｜把业务权重归一化。** $\alpha_i=w_i/\sum_jw_j$，于是 $\sum_i\alpha_i=1$。

**第 3 步｜加权误差是误差的加权平均。** 

$$
WMAE=\sum_i\alpha_ie_i=\frac{\sum_iw_i|y_i-\hat y_i|}{\sum_iw_i}
$$

权重越大的客户对最终指标影响越大，所以权重规则必须在建模前确定并可审计。

**把上面的关系收束为本章计算式：**

$$
WMAE=\frac{\sum_iw_i|y_i-\hat{y}_i|}{\sum_iw_i}
$$

**符号解释：** $w_i$ 可表示客户金额、业务优先级或样本权重。

**代码对应：** 同时报告普通 MAE 与业务加权误差，并检查高价值客户误差。

**使用边界：** 权重体现决策偏好，必须由业务规则给出，不能为提高分数随意调整。


In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"]
)
audit = pd.Series(
    {
        "原始行": len(raw),
        "完全重复": raw.duplicated().sum(),
        "客户缺失": raw.CustomerID.isna().sum(),
        "取消单": raw.InvoiceNo.astype(str).str.startswith("C").sum(),
        "数量非正": (raw.Quantity <= 0).sum(),
        "单价非正": (raw.UnitPrice <= 0).sum(),
    }
)
print(audit.to_string())
print("日期范围:", raw.InvoiceDate.min(), "至", raw.InvoiceDate.max())
display(raw.head())


## 练一练：审计一份客户消费订单

对照上面 108.5 原始数据质量审计的做法，对下面这份**内置示例订单表**做同样的体检。为让你不依赖外部大文件 `datasets/uci_online_retail_200k.csv`，这里直接用一小段字段与真实交易一致的模拟数据（InvoiceNo / CustomerID / Quantity / UnitPrice）。

请在下方代码里补全空白，统计出这张订单表的：**①完全重复行数 ②客户编号缺失行数 ③取消单数量（发票号以 C 开头）**，并让自检通过。


In [ ]:
# 请在下方填写代码
import numpy as np
import pandas as pd

# ① 完全重复行数（提示：orders.duplicated().sum()）
duplicates = None  # TODO
# ② 客户编号缺失行数（提示：orders.CustomerID.isna().sum()）
missing_customer = None  # TODO
# ③ 取消单数量：发票号转成字符串后以 "C" 开头（提示：str.startswith("C")）
cancellations = None  # TODO


In [ ]:
# 完整答案
import numpy as np
import pandas as pd

orders = pd.DataFrame(
    {
        "InvoiceNo": ["A001", "A002", "A002", "C003", "A004", None],
        "CustomerID": [1001.0, 1002.0, 1002.0, 1003.0, None, 1005.0],
        "Quantity": [2, 1, 1, -4, 5, 2],
        "UnitPrice": [10.0, 20.0, 20.0, 5.0, 8.0, 12.0],
    }
)

# ① 完全重复行数
duplicates = orders.duplicated().sum()
# ② 客户编号缺失行数
missing_customer = orders.CustomerID.isna().sum()
# ③ 取消单数量：发票号以 "C" 开头
cancellations = orders.InvoiceNo.astype(str).str.startswith("C").sum()


## 116.10 清洗交易并记录样本变化

删除重复后，仅保留有客户编号的有效正向销售，并计算清洗保留率。


In [ ]:
dedup = raw.drop_duplicates().copy()
valid = (
    (~dedup.InvoiceNo.astype(str).str.startswith("C"))
    & (dedup.Quantity > 0)
    & (dedup.UnitPrice > 0)
    & dedup.CustomerID.notna()
)
sales = dedup.loc[valid].copy()
sales["revenue"] = sales.Quantity * sales.UnitPrice
clean_report = pd.Series(
    {
        "去重后": len(dedup),
        "有效销售": len(sales),
        "删除行": len(raw) - len(sales),
        "保留率": len(sales) / len(raw),
    }
)
print(clean_report.round(3).to_string())
print(
    "单行金额分位数:\n",
    sales.revenue.quantile([0.5, 0.9, 0.99, 0.999]).round(2),
)


## 116.11 定义时间窗口并构造客户样本

截止日前的数据只生成特征，截止日后的数据只生成目标，从源头防止时间穿越。


In [ ]:
cutoff = sales.InvoiceDate.quantile(0.8).normalize()
hist = sales[sales.InvoiceDate < cutoff]
future = sales[sales.InvoiceDate >= cutoff]
customer_features = hist.groupby("CustomerID").agg(
    recency=("InvoiceDate", lambda x: (cutoff - x.max().normalize()).days),
    frequency=("InvoiceNo", "nunique"),
    monetary=("revenue", "sum"),
    items=("Quantity", "sum"),
    products=("StockCode", "nunique"),
    active_days=(
        "InvoiceDate",
        lambda x: (x.max().normalize() - x.min().normalize()).days + 1,
    ),
)
future_target = (
    future.groupby("CustomerID").revenue.sum().rename("future_revenue")
)
customer = customer_features.join(future_target, how="left").fillna(
    {"future_revenue": 0}
)


## 116.12 探索目标分布与客户差异

金额目标同时具有大量零值和明显长尾，后续需要在对数尺度训练、原始金额尺度评价。


In [ ]:
target_summary = customer.future_revenue.describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
).round(2)
customer["history_value_group"] = pd.qcut(
    customer.monetary, 4, labels=["Q1", "Q2", "Q3", "Q4"]
)
group_summary = customer.groupby("history_value_group", observed=True).agg(
    customers=("future_revenue", "size"),
    future_positive_rate=("future_revenue", lambda x: (x > 0).mean()),
    future_mean=("future_revenue", "mean"),
    future_median=("future_revenue", "median"),
)
print(target_summary)
display(group_summary.round(2))


## 116.13 客户级划分与Dummy基线

按客户划分训练集和测试集，并用中位数回归器建立最低比较基线。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

cols = ["recency", "frequency", "monetary", "items", "products", "active_days"]
X = customer[cols].clip(lower=0)
y_raw = customer.future_revenue
y = np.log1p(y_raw)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=105, stratify=(y_raw > 0)
)
dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
print("训练/测试客户:", len(X_train), len(X_test))
print(
    "训练/测试未来有消费比例:",
    f"{(y_train > 0).mean():.1%}",
    f"{(y_test > 0).mean():.1%}",
)


## 116.14 比较线性模型与树模型

使用五折交叉验证比较Ridge与梯度提升，以对数目标MAE选择模型。


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "梯度提升": HistGradientBoostingRegressor(
        max_iter=180, max_leaf_nodes=12, l2_regularization=2, random_state=105
    ),
}
cv = KFold(5, shuffle=True, random_state=105)
rows = []
for name, model in models.items():
    scores = -cross_val_score(
        model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
    )
    rows.append([name, scores.mean(), scores.std()])
validation = pd.DataFrame(
    rows, columns=["model", "CV_log_MAE", "std"]
).sort_values("CV_log_MAE")
display(validation.round(3))
best_name = validation.iloc[0].model
best_model = models[best_name].fit(X_train, y_train)


## 116.15 在原始金额尺度评价模型

把预测还原为金额，联合报告MAE、RMSE、R²和Dummy基线误差。


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

prediction = np.maximum(0, np.expm1(best_model.predict(X_test)))
actual = np.expm1(y_test)
baseline = np.maximum(0, np.expm1(dummy.predict(X_test)))
metrics = pd.Series(
    {
        "MAE_GBP": mean_absolute_error(actual, prediction),
        "RMSE_GBP": mean_squared_error(actual, prediction) ** 0.5,
        "R2": r2_score(actual, prediction),
        "Dummy_MAE": mean_absolute_error(actual, baseline),
    }
)
result = X_test.copy()
result["actual"] = actual.to_numpy()
result["prediction"] = prediction
result["absolute_error"] = (result.actual - result.prediction).abs()
print("最佳模型:", best_name)
print(metrics.round(2).to_string())


## 116.16 使用Top-K检查排序能力

Top-K只作为模型排序评价，比较不同观察比例覆盖了多少真实未来消费金额。


In [ ]:
ranked = result.sort_values("prediction", ascending=False)
total_revenue = max(ranked.actual.sum(), 1e-9)
topk_rows = []
for share in [0.05, 0.10, 0.20]:
    n = max(1, int(len(ranked) * share))
    topk_rows.append(
        [
            f"{share:.0%}",
            n,
            ranked.head(n).actual.sum() / total_revenue,
            ranked.head(n).actual.mean(),
        ]
    )
topk = pd.DataFrame(
    topk_rows, columns=["观察比例", "客户数", "真实金额覆盖率", "组内平均金额"]
)
display(topk.round(3))


## 116.17 错误切片与高误差案例

分别检查未来零消费/有消费客户，以及不同历史价值组的误差。


In [ ]:
error_table = result.copy()
error_table["future_status"] = np.where(
    error_table.actual > 0, "未来有消费", "未来零消费"
)
error_table["history_quartile"] = (
    pd.qcut(error_table.monetary, 4, labels=False, duplicates="drop") + 1
)
status_error = error_table.groupby("future_status").absolute_error.agg(
    ["count", "mean", "median"]
)
value_error = error_table.groupby("history_quartile").absolute_error.agg(
    ["count", "mean", "median"]
)
print("按未来状态误差:\n", status_error.round(2))
print("按历史价值分组误差:\n", value_error.round(2))
display(
    error_table.nlargest(8, "absolute_error")[
        ["actual", "prediction", "absolute_error", "monetary", "frequency"]
    ].round(2)
)


## 116.18 特征解释与模型局限

置换重要性说明模型依赖哪些历史信号，不能据此断言这些变量会导致未来消费。


In [ ]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=5,
    scoring="neg_mean_absolute_error",
    random_state=105,
)
importance = pd.Series(permutation.importances_mean, index=cols).sort_values(
    ascending=False
)
print("置换重要性:\n", importance.round(4))
print(
    "局限: 固定交易样本、客户编号缺失、未来零值多且金额长尾；预测关系不代表营销干预的因果效果。"
)


## 116.19 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 116.19.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 116.19.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 116.20 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 116.20.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 116.21 易错点提醒

**易错点 1**：客户价值标签用"全期金额"计算时，特征里不能混入未来信息（如整期汇总特征），否则标签泄漏、上线即失效。

**易错点 2**：预测时点（如按月快照）不统一，有的客户窗口长、有的短，价值不可比；先定死观测截止日与未来窗口。

**易错点 3**：重复订单与取消单（C 开头 InvoiceNo）不清理就进特征，金额与频次都会被污染。

**易错点 4**：CustomerID 缺失行丢弃后要报告占比，避免样本只覆盖"老客户"造成选择偏差。

**易错点 5**：金额单位不统一（英镑/美元）直接混合建模，模型学到的"价值"没有业务意义。


## 116.22 结论与表达

- 客户级预测必须先定义观察窗口和未来窗口
- 金额长尾需要区分训练尺度与评价尺度
- 总体误差、Top-K和分组误差回答不同问题
- 特征重要性反映预测依赖而不是因果关系


## 116.23 项目验收清单

- 完成原始审计与清洗报告
- 观察和目标窗口无重叠
- 比较Dummy与两个候选模型
- 报告MAE、RMSE、R²和Top-K
- 完成错误切片与置换重要性解释

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 116.24 小结

使用 UCI Online Retail 公开交易数据，按照“问题定义—数据准备—模型训练—模型评价—模型理解”的教学路径，预测客户未来消费金额。


### 116.24.1 你已经完成

- 理解客户价值预测的样本粒度与时间窗口
- 审计并清洗真实交易数据
- 构造无时间穿越的客户特征与目标
- 比较回归基线、线性模型和树模型
- 使用金额误差、Top-K与错误切片理解模型


### 116.24.2 质量与结论提醒

- 重复、取消、退货和非正价格
- CustomerID缺失和清洗保留率
- 观察窗口与目标窗口严格分离
- 客户级预测必须先定义观察窗口和未来窗口
- 金额长尾需要区分训练尺度与评价尺度
- 总体误差、Top-K和分组误差回答不同问题
- 特征重要性反映预测依赖而不是因果关系


### 116.24.3 学习检查

- [ ] 完成原始审计与清洗报告
- [ ] 观察和目标窗口无重叠
- [ ] 比较Dummy与两个候选模型
- [ ] 报告MAE、RMSE、R²和Top-K
- [ ] 完成错误切片与置换重要性解释


### 116.24.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
